In [33]:
import os
import sys
from glob import glob

import numpy as np
import pandas as pd

sys.path.append("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code")
from project_utils.pet.helper import check_file_exists

sys.path.append("/home/b.y.yang/sotiraslab/BradenFunctions")
from byypy.nifti import *

In [34]:
odir = "/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/exploratory/a4_treatment/pet_analysis"
pet_analysis_odir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/pet_analysis_A4_treatment/nipype"
create_pet_json_wdir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/pet_analysis_A4_treatment/_petjson"

In [35]:
data_table = pd.read_csv("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/exploratory/a4_treatment/pet_analysis/petanalysis.csv")

In [37]:
a4_idx = data_table["site"] == "A4"
adni_fbp_idx = (data_table["site"] == "ADNI") & (data_table["tracer"] == "FBP")
adni_fbb_idx = (data_table["site"] == "ADNI") & (data_table["tracer"] == "FBB")
oasis_pib_idx = (data_table["site"] == "OASIS") & (data_table["tracer"] == "PIB")
oasis_fbp_idx = (data_table["site"] == "OASIS") & (data_table["tracer"] == "FBP")
habs_idx = data_table["site"] == "HABS"
mcsa_idx = data_table["site"] == "MCSA"

# pac_idx = data_table["site_in_pac"]
aibl_idx = data_table["site"] == "AIBL"
biocard_idx = data_table["site"] == "BIOCARD"
blsa_idx = data_table["site"] == "BLSA"
wrap_idx = data_table["site"] == "WRAP"

# Fill in missing JSON files with dummy paths

In [5]:
# def get_dummy_json_path(row):
#     return os.path.join(create_pet_json_wdir, f"{row['idvi']}.json")
# pet_json_nan = data_table["filepath.raw_amyloid.json"].isna()
# data_table.loc[pet_json_nan,"filepath.raw_amyloid.json"] = data_table.loc[pet_json_nan,:].apply(get_dummy_json_path, axis = 1)

# Preprocessing

### Get number of frames for each scan

In [38]:
nifti_checker = NiftiChecker()
data_table["num_frames"] = data_table["filepath.raw_amyloid"].map(nifti_checker.count_frames_single)

### Check that files exists

In [41]:
exists_idx, _ = check_file_exists(data_table, ["filepath.raw_amyloid", "musemaskpath", "musemriskullpath", "muselabelpath"])

In [42]:
missing_table = data_table[~exists_idx]
data_table = data_table[exists_idx]

In [43]:
# missing_table.to_csv("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/exploratory/a4_treatment/pet_analysis/missing_20250620.csv", index = False)

In [44]:
data_table_copy = data_table.copy()

In [45]:
# data_table_filter = data_table[exists_idx]

# Get options

### t_start, t_end

In [46]:
data_table.loc[:,"arg__t_start"], data_table.loc[:,"arg__t_end"] = "--t_start 0", "--t_end 99999"

data_table.loc[oasis_pib_idx,"arg__t_start"] = "--t_start 40"
data_table.loc[oasis_pib_idx,"arg__t_end"] = "--t_end 60"

data_table.loc[oasis_fbp_idx,"arg__t_start"] = "--t_start 50"
data_table.loc[oasis_fbp_idx,"arg__t_end"] = "--t_end 70"

data_table.loc[biocard_idx|blsa_idx|wrap_idx,"arg__t_start"] = "--t_start 40"
data_table.loc[biocard_idx|blsa_idx|wrap_idx,"arg__t_end"] = "--t_end 60"

/tmp/ipykernel_2639839/1078890213.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_2639839/1078890213.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### realign_t_start, realign_t_end

In OASIS-3, some FBP and PiB scans were only acquired starting at the averaging window of 50 min and 30 min (respectively), and the scan is missing the frames before the window. Therefore, we will handle the frame alignment a little bit differently for these cases, by not computing the reference scan and instead just using the first frame of the image as the reference.

We'll assume that for these FBP scans they only have 4 frames, and for these PiB scans they only have 6 frames.

In [47]:
data_table.loc[:,"arg__realign_t_start"], data_table.loc[:,"arg__realign_t_end"] = "", ""

data_table.loc[oasis_pib_idx & (data_table["num_frames"] > 6),"arg__realign_t_start"] = "--realign_t_start 30"
data_table.loc[oasis_pib_idx & (data_table["num_frames"] > 6),"arg__realign_t_end"] = "--realign_t_end 40"

data_table.loc[oasis_fbp_idx & (data_table["num_frames"] > 4), "arg__realign_t_start"] = "--realign_t_start 40"
data_table.loc[oasis_fbp_idx & (data_table["num_frames"] > 4), "arg__realign_t_end"] = "--realign_t_end 50"

data_table.loc[biocard_idx|blsa_idx|wrap_idx,"arg__realign_t_start"] = "--realign_t_start 30"
data_table.loc[biocard_idx|blsa_idx|wrap_idx,"arg__realign_t_end"] = "--realign_t_end 40"

/tmp/ipykernel_2639839/217673002.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_2639839/217673002.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### pet_is_3d

In [48]:
data_table.loc[:,"arg__pet_is_3d"] = ""

data_table.loc[adni_fbp_idx,"arg__pet_is_3d"] = "--pet_is_3d"
data_table.loc[aibl_idx,"arg__pet_is_3d"] = "--pet_is_3d"

/tmp/ipykernel_2639839/3946478966.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### reg_smooth_fwhm

In [49]:
data_table.loc[:,"arg__reg_smooth_fwhm"] = ""

data_table.loc[adni_fbp_idx,"arg__reg_smooth_fwhm"] = "--reg_smooth_fwhm '0.01'"  # ADNI PET is already smoothed so no need for more smoothing

/tmp/ipykernel_2639839/2705846559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### modify_json_func

In [50]:
data_table.loc[:,"arg__modify_json_func"] = ""

data_table.loc[a4_idx, "arg__modify_json_func"] = "--modify_json_func 'modify_json_add_FrameTimesStart'"
data_table.loc[(oasis_fbp_idx | oasis_pib_idx), "arg__modify_json_func"] = "--modify_json_func 'modify_json_change_FrameTimes_to_Time'"
data_table.loc[mcsa_idx, "arg__modify_json_func"] = "--modify_json_func 'modify_json_add_FrameTimesStart'"

/tmp/ipykernel_2639839/1208752207.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### create_pet_json

In [51]:
data_table.loc[:,"arg__create_pet_json"] = ""
data_table.loc[:,"arg__create_pet_json_n_frames"] = ""
data_table.loc[:,"arg__create_pet_json_frame_duration_min"] = ""

data_table.loc[habs_idx, "arg__create_pet_json"] = "--create_pet_json"
data_table.loc[habs_idx, "arg__create_pet_json_n_frames"] = "--create_pet_json_n_frames 10"
data_table.loc[habs_idx, "arg__create_pet_json_frame_duration_min"] = "--create_pet_json_frame_duration_min 2"

/tmp/ipykernel_2639839/3698381724.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_2639839/3698381724.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/tmp/ipykernel_2639839/3698381724.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### iterative_smoothing

In [52]:
data_table.loc[:,"arg__iterative_smoothing"] = "--iterative_smoothing"

data_table.loc[adni_fbp_idx,"arg__iterative_smoothing"] = ""  # ADNI PET is already smoothed so no need for more smoothing

/tmp/ipykernel_2639839/1981375620.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


### manual correction of PET/MRI registration

In [18]:
manual_correct_dir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/pet_analysis/manual_correct"
manual_correct_paths = glob(os.path.join(manual_correct_dir, "*.mat"))
subj_to_manual_correct = pd.Series(manual_correct_paths) \
    .str.split("/", expand = True).iloc[:,-1] \
    .str.replace(".mat","")
subj_to_manual_correct_idx = data_table["idvi"].isin(subj_to_manual_correct)

data_table.loc[:,"arg__manual_coreg_mat"] = ""
data_table.loc[subj_to_manual_correct_idx,"arg__manual_coreg_mat"] = f"--manual_coreg_mat {manual_correct_dir}/" + data_table.loc[subj_to_manual_correct_idx,"idvi"] + ".mat"

# data_table.loc[subj_to_manual_correct_idx,"arg__iterative_smoothing"] = "--manual_coreg_mat " + pd.Series(manual_correct_paths)
# subj_to_manual_correct_df = pd.DataFrame(subj_to_manual_correct, columns = ["manual_coreg_mat"])
# subj_to_manual_correct_df["idvi"] = subj_to_manual_correct_df.iloc[:,-1] \
#     .str.split("/", expand = True).iloc[:,-1] \
#     .str.replace(".mat","")
# manual_correct_cmd = pd.merge(subj_to_manual_correct_df, data_table_cmd, on = "idvi", how = "left")
# manual_correct_cmd["cmd"] = manual_correct_cmd["cmd"] + " --manual_coreg_mat " + manual_correct_cmd["manual_coreg_mat"] + " --overwrite"

# Create commands

In [53]:
def create_commands(data_table):
    cmd = "~/miniconda3/envs/ad/bin/python /home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/2_Preprocessing/compute_suvr.py -i " \
        + data_table["idvi"] \
        + " --petpath " + data_table["filepath.raw_amyloid"] \
        + " --petjsonfile " + data_table["filepath.raw_amyloid.json"] \
        + " --musemaskpath " + data_table["musemaskpath"] \
        + " --musemriskullpath " + data_table["musemriskullpath"] \
        + " --muselabelpath " + data_table["muselabelpath"] \
        + f" -o {pet_analysis_odir}"
    additional_options = data_table.filter(regex="^arg__").agg(" ".join, axis = 1)
    cmd = cmd + " " + additional_options
    cmd.name = "cmd"
    
    return pd.concat([data_table, cmd], axis = 1)

In [54]:
data_table_cmd = create_commands(data_table)

### for now, remove ADNI FBB from analysis

In [55]:
data_table_cmd = data_table_cmd[~adni_fbb_idx]

/tmp/ipykernel_2639839/1904049860.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


# Save commands

In [61]:
pd.DataFrame(data_table_cmd["cmd"]).to_csv(os.path.join(odir, "cmd.txt"), header = False, index = False)
data_table_cmd.to_csv(os.path.join(odir, "data_table.csv"), index = False, sep = ",")
missing_table.to_csv(os.path.join(odir, "missing.csv"), index = False, sep = ",")

### Commands with --overwrite

In [21]:
data_table_overwrite = data_table_cmd.copy()
data_table_overwrite.loc[:,"cmd"] = data_table_overwrite["cmd"] + " --overwrite"
pd.DataFrame(data_table_overwrite["cmd"]).to_csv(os.path.join(odir, "cmd_overwrite.txt"), header = False, index = False)